In [18]:
import os
import requests
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import mlflow
from typing import Dict, Tuple, List
from dataclasses import dataclass
DATA_PATH = "../../02_data/mnist"
NUM_CLIENTS = 3
NUM_ATTACERS = 1
NUM_NORMAL_CLIENTS = NUM_CLIENTS - NUM_ATTACERS

mlflow.set_tracking_uri("http://localhost:5100")
MLFLOW_EXPERIMENT_NAME = "edge-fl-experiment"

In [2]:
def pull_test_data():
    x_test_binary = requests.get("http://localhost:8555/test-data/test_images.npy")
    x_test_binary.raise_for_status()
    y_test_binary = requests.get("http://localhost:8555/test-data/test_labels.npy")
    y_test_binary.raise_for_status()
    with open("../../02_data/mnist/test_images.npy", "wb") as f:
        f.write(x_test_binary.content)
    with open("../../02_data/mnist/test_labels.npy", "wb") as f:
        f.write(y_test_binary.content)
    x_test = np.load("../../02_data/mnist/test_images.npy", allow_pickle=True)
    y_test = np.load("../../02_data/mnist/test_labels.npy", allow_pickle=True)
    return x_test, y_test

x_test, y_test = pull_test_data()

np.savez(
    "../../02_data/agg_test/course_test.npz",
    x_test=x_test,
    y_test=y_test,
)

In [3]:
!docker run --name aggregator_test -d --rm \
    -p 5100:5000 -p 9100:8000 \
    -v "$(realpath ../../02_data/agg_test):/data" \
    ghcr.io/bencefarkas27/aggregator:test-auth

docker: Error response from daemon: Conflict. The container name "/aggregator_test" is already in use by container "a4cc8a6aa37c5e4c541c5cd81153fb851a020b01f95a84cf770b4b8f3da8e282". You have to remove (or rename) that container to be able to reuse that name.

Run 'docker run --help' for more information


In [4]:
response = requests.get("http://localhost:9100/health")
print(f"Health check response of aggregator: {response.status_code} - {response.text}")

Health check response of aggregator: 200 - 


In [5]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "admin"
with open("../../04_deploy/aggregator/cfg/mlflow_admin.json", "r") as f:
                admin_config = json.load(f)
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["admin_password"]
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1788428945789, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788428945789, lifecycle_stage='active', name='edge-fl-experiment', tags={}, trace_location=None, workspace='default'>

In [6]:
def create_dummy_model(input_shape):
    model = keras.Sequential([
        keras.layers.InputLayer(shape=input_shape),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    return model

def mlflow_load_model(model_name):
    response = requests.get(f"http://localhost:8555/{model_name}.weights.h5")
    response.raise_for_status()
    weights_path = f"{model_name}.weights.h5"
    with open(weights_path, "wb") as f:
        f.write(response.content)
    input_shape = os.environ.get("MLFLOW_INPUT_SHAPE", (28, 28))
    model = create_dummy_model(input_shape=input_shape)
    model.load_weights(weights_path)
    model.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])
    with mlflow.start_run(run_name=f"{model_name}_run") as run:
        # Log the model to MLflow
        mlflow.tensorflow.log_model(model, f"{model_name}_model")
        return run.info.run_id

def get_run_ids():
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    runs_df = mlflow.search_runs([experiment.experiment_id])
    run_ids = runs_df["run_id"].tolist()
    run_ids.reverse()
    return run_ids


In [7]:
mlflow_load_model("aggregated")
for client_id in range(NUM_CLIENTS):
        mlflow_load_model(f"client_{client_id}")
run_ids = get_run_ids()
for client_id in range(NUM_CLIENTS):
        x_train_binary = requests.get(f"http://localhost:8555/client_data/client_{client_id}_train_images.npy")
        with open(f"../../02_data/mnist/client_data/client_{client_id}_train_images.npy", "wb") as f:
            f.write(x_train_binary.content)

2026/09/03 11:49:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 11:49:36 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run aggregated_run at: http://localhost:5100/#/experiments/1/runs/fd9b77318d5d4dd49bff5884647569e5
🧪 View experiment at: http://localhost:5100/#/experiments/1


2026/09/03 11:49:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 11:49:43 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_0_run at: http://localhost:5100/#/experiments/1/runs/5a2863f7848c436d9e66176a80dba4dd
🧪 View experiment at: http://localhost:5100/#/experiments/1


2026/09/03 11:49:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 11:49:50 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_1_run at: http://localhost:5100/#/experiments/1/runs/08a4248ca958467aaecb84f30977a798
🧪 View experiment at: http://localhost:5100/#/experiments/1


2026/09/03 11:49:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 11:49:58 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_2_run at: http://localhost:5100/#/experiments/1/runs/21eb45b09358400db43b2178fec5306b
🧪 View experiment at: http://localhost:5100/#/experiments/1


In [8]:
def call_do_agg_with_deltarep():
    client_ids = run_ids[1:]
    aggregation_config = {
        "local_model_hosts": [
            "http://localhost:5000"
            for _ in range(NUM_CLIENTS)
        ],
        "local_model_uris": [
            f"runs:/{client_ids[client_id]}/client_{client_id}_model"
            for client_id in range(NUM_CLIENTS)
        ],
        "local_data_size": [
            len(np.load(f"../../02_data/mnist/client_data/client_{client_id}_train_images.npy"))
            for client_id in range(NUM_CLIENTS)
        ],
        "aggregation_algorithm": "deltarep",
        "global_model_host": "http://localhost:5000",
        "global_model_uri": f"runs:/{run_ids[0]}/aggregated_model"            
    }
    response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)
    response.raise_for_status()
    return response
    

In [9]:
response = call_do_agg_with_deltarep()
print(response.json())

{'aggregated_model_host_external': 'http://localhost:5100', 'aggregated_model_host_organizational': 'http://aggregator:5000', 'aggregated_model_uri': 'models:/m-e8e4af4f20dd4018a466c625a3be4905', 'aggregated_model_username': 'model_user', 'aggregated_model_password': 'hiPMMENYuzajkO_vuq9owKnV3ORxRSapm7GgpNw9BAU'}


In [10]:
def pull_and_eval(prev_agg_id, loop_cycle=""):
    # Search the experiment and retrieve the runs
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    models = mlflow.search_logged_models(
        experiment_ids=[experiment.experiment_id],
        order_by=[{"field_name": "creation_time", "ascending": False}],
        output_format="list"
    )
    new_aggregated = models[0]
    new_aggregated_model = mlflow.tensorflow.load_model(f"models:/{new_aggregated.model_id}")
    new_aggregated_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    run_name = f"runs:/{prev_agg_id}/deltarep_cycle{loop_cycle}_model" if loop_cycle else f"runs:/{prev_agg_id}/aggregated_model"
    previous_aggregated_model = mlflow.tensorflow.load_model(run_name)
    previous_aggregated_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    new_aggregated_loss, new_aggregated_accuracy = new_aggregated_model.evaluate(x_test, y_test, verbose=0)
    print(f"New Aggregated Model - Loss: {new_aggregated_loss}, Accuracy: {new_aggregated_accuracy}")

    previous_aggregated_loss, previous_aggregated_accuracy = previous_aggregated_model.evaluate(x_test, y_test, verbose=0)
    print(f"Previous Aggregated Model - Loss: {previous_aggregated_loss}, Accuracy: {previous_aggregated_accuracy}")

    return new_aggregated_model, previous_aggregated_model

In [11]:
new_aggregated_model, previous_aggregated_model = pull_and_eval(get_run_ids()[0])

/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


New Aggregated Model - Loss: 11.982089042663574, Accuracy: 0.22759999334812164
Previous Aggregated Model - Loss: 37.268497467041016, Accuracy: 0.2921000123023987


In [12]:
new_aggregated_model.save_weights("dummy_models/deltarep_cycle1.weights.h5")
with mlflow.start_run(run_name=f"deltarep_cycle1") as run:
        # Log the model to MLflow
        mlflow.tensorflow.log_model(new_aggregated_model, f"deltarep_cycle1_model")
print(run.info.run_id)

2026/09/03 11:57:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 11:57:17 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run deltarep_cycle1 at: http://localhost:5100/#/experiments/1/runs/7d7ffb4137e149129f1588cf4d9fa9dd
🧪 View experiment at: http://localhost:5100/#/experiments/1
7d7ffb4137e149129f1588cf4d9fa9dd


In [25]:
# -----------------------------
# Attacks on model updates (delta)
# -----------------------------
WeightList = List[np.ndarray]
def sign_flip(delta: WeightList) -> WeightList:
    return [(-weights).copy() for weights in delta]

def scaled_sign_flip(delta: WeightList, scale_factor: float) -> WeightList:
    return [(-weights * scale_factor).copy() for weights in delta]

def add_noise(delta: WeightList, sigma: float) -> WeightList:
    return [
        (weights + sigma * np.random.randn(*weights.shape)).astype(weights.dtype)
        for weights in delta
    ]
@dataclass
class AttackSpec:
    attack_type: str
    scale_factor: float = 5.0 # gamma
    noise_level: float = 0.1  # sigma

class AttackManager:
    def __init__(self, client_specs: Dict[int, AttackSpec]):
        self.client_specs = client_specs
    def apply(self, client_id: int, delta: WeightList, round: int) -> Tuple[WeightList, str]:
        spec = self.client_specs.get(client_id, AttackSpec("none"))
        type = spec.attack_type.lower()
        if type == "none":
            return delta, "none"
        attacked_delta = [w.copy() for w in delta]
        if type == "sign_flip":
            attacked_delta = sign_flip(attacked_delta)
        elif type == "scaled":
            attacked_delta = scaled_sign_flip(attacked_delta, spec.scale_factor)
        elif type == "random_noise":
            attacked_delta = add_noise(attacked_delta, spec.noise_level)

        return attacked_delta, type

In [26]:
def retrain_clients_on_new_aggregate(loop_cycle, run_id, attack_manager):
    for client_id in range(NUM_CLIENTS):
        # Load the new aggregated model weights into the client model
        client_model = mlflow.tensorflow.load_model(f"runs:/{run_id}/deltarep_cycle{loop_cycle}_model")
        weights_before_training = [
            weights.copy()
            for weights in client_model.get_weights()
        ]
        x_train_binary = requests.get(f"http://localhost:8555/client_data/client_{client_id}_train_images.npy")
        with open(f"../../02_data/mnist/client_data/client_{client_id}_train_images.npy", "wb") as f:
            f.write(x_train_binary.content)
        y_train_binary = requests.get(f"http://localhost:8555/client_data/client_{client_id}_train_labels.npy")
        with open(f"../../02_data/mnist/client_data/client_{client_id}_train_labels.npy", "wb") as f:
            f.write(y_train_binary.content)
        x_train = np.load(f"../../02_data/mnist/client_data/client_{client_id}_train_images.npy", allow_pickle=True)
        y_train = np.load(f"../../02_data/mnist/client_data/client_{client_id}_train_labels.npy", allow_pickle=True)
        client_model.compile(optimizer='adam',
                             loss='sparse_categorical_crossentropy',
                             metrics=['accuracy'])
        client_model.fit(x_train, y_train, epochs=3)
        delta = [after - before
                 for after, before in zip(client_model.get_weights(), weights_before_training)
        ]
        delta, attack_type = attack_manager.apply(client_id, delta, loop_cycle)
        final_weights = [before + d for before, d in zip(weights_before_training, delta)]
        client_model.set_weights(final_weights)
        with mlflow.start_run(run_name=f"client_{client_id}_run_{loop_cycle}") as run:
                # Log the model to MLflow
                mlflow.tensorflow.log_model(client_model, f"client_{client_id}_model_{loop_cycle}")

        client_model.save_weights(f"dummy_models/cycle{loop_cycle}/client_{client_id}.weights.h5")  


def rerun_deltarep_aggregation(loop_cycle):
    run_ids = get_run_ids()[-4:]
    print(run_ids)
    clients_ids = run_ids[1:]
    aggregation_config = {
            "local_model_hosts": [
                "http://localhost:5000"
                for _ in range(NUM_CLIENTS)
            ],
            "local_model_uris": [
                f"runs:/{clients_ids[client_id]}/client_{client_id}_model_{loop_cycle}"
                for client_id in range(NUM_CLIENTS)
            ],
            "local_data_size": [
                len(np.load(f"../../02_data/mnist/client_data/client_{client_id}_train_images.npy"))
                for client_id in range(NUM_CLIENTS)
            ],
            "aggregation_algorithm": "deltarep",
            "global_model_host": "http://localhost:5000",
            "global_model_uri": f"runs:/{run_ids[0]}/deltarep_cycle{loop_cycle}_model"            
        }
    response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)
    response.raise_for_status()

In [27]:
attack_manager = AttackManager({
        0: AttackSpec("none"),
        1: AttackSpec("none"),
        2: AttackSpec("sign_flip")
    })

In [28]:
prev_agg_id = get_run_ids()[-1]
retrain_clients_on_new_aggregate(loop_cycle=1, run_id=prev_agg_id, attack_manager=attack_manager)
rerun_deltarep_aggregation(loop_cycle=1)
new_aggregated_model, previous_aggregated_model = pull_and_eval(prev_agg_id, loop_cycle=1)

new_aggregated_model.save_weights("dummy_models/deltarep_cycle2.weights.h5")
with mlflow.start_run(run_name=f"deltarep_cycle2") as run:
        # Log the model to MLflow
        mlflow.tensorflow.log_model(new_aggregated_model, f"deltarep_cycle2_model")
print(run.info.run_id)

Epoch 1/3
768/768 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9782 - loss: 0.1552
Epoch 2/3
768/768 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9881 - loss: 0.0634
Epoch 3/3
768/768 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9911 - loss: 0.0391


2026/09/03 12:30:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 12:30:33 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_0_run_1 at: http://localhost:5100/#/experiments/1/runs/562dcdd5cd254c9ca6089e17f5199779
🧪 View experiment at: http://localhost:5100/#/experiments/1
Epoch 1/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9428 - loss: 0.3135
Epoch 2/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9818 - loss: 0.0588
Epoch 3/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9861 - loss: 0.0448


2026/09/03 12:30:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 12:30:48 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_1_run_1 at: http://localhost:5100/#/experiments/1/runs/66c7b2d0e8d24f65a5487a922d4518ca
🧪 View experiment at: http://localhost:5100/#/experiments/1
Epoch 1/3
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9650 - loss: 0.2245
Epoch 2/3
564/564 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9891 - loss: 0.0475
Epoch 3/3
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9926 - loss: 0.0318


2026/09/03 12:31:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 12:31:04 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_2_run_1 at: http://localhost:5100/#/experiments/1/runs/e2f805fe6f1d40b781d8bea49712db17
🧪 View experiment at: http://localhost:5100/#/experiments/1
['7d7ffb4137e149129f1588cf4d9fa9dd', '562dcdd5cd254c9ca6089e17f5199779', '66c7b2d0e8d24f65a5487a922d4518ca', 'e2f805fe6f1d40b781d8bea49712db17']
New Aggregated Model - Loss: 10.084258079528809, Accuracy: 0.4077000021934509
Previous Aggregated Model - Loss: 11.982089042663574, Accuracy: 0.22759999334812164


2026/09/03 12:31:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 12:31:38 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run deltarep_cycle2 at: http://localhost:5100/#/experiments/1/runs/bead6e8f81b141669b6cc7ce9720dde9
🧪 View experiment at: http://localhost:5100/#/experiments/1
bead6e8f81b141669b6cc7ce9720dde9


In [29]:
prev_agg_id = get_run_ids()[-1]
retrain_clients_on_new_aggregate(loop_cycle=2, run_id=prev_agg_id, attack_manager=attack_manager)
rerun_deltarep_aggregation(loop_cycle=2)
new_aggregated_model, previous_aggregated_model = pull_and_eval(prev_agg_id, loop_cycle=2)
new_aggregated_model.save_weights("dummy_models/deltarep_cycle3.weights.h5")
with mlflow.start_run(run_name=f"deltarep_cycle3") as run:
        # Log the model to MLflow
        mlflow.tensorflow.log_model(new_aggregated_model, f"deltarep_cycle3_model")
print(run.info.run_id)

/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_15', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Epoch 1/3
768/768 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9862 - loss: 0.0937
Epoch 2/3
768/768 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9905 - loss: 0.0475
Epoch 3/3
768/768 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9913 - loss: 0.0455


2026/09/03 12:35:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 12:36:00 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_0_run_2 at: http://localhost:5100/#/experiments/1/runs/6e1ce5e94ef14dc58669f701be0c2e72
🧪 View experiment at: http://localhost:5100/#/experiments/1
Epoch 1/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9556 - loss: 0.2028
Epoch 2/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9824 - loss: 0.0615
Epoch 3/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9858 - loss: 0.0460


2026/09/03 12:36:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/03 12:36:14 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


...

🏃 View run client_2_run_2 at: http://localhost:5100/#/experiments/1/runs/be8ede5334804431b7b40391599cb1c5
🧪 View experiment at: http://localhost:5100/#/experiments/1
['bead6e8f81b141669b6cc7ce9720dde9', '6e1ce5e94ef14dc58669f701be0c2e72', 'd0705b55ff034bcda62b5b70bfa568db', 'be8ede5334804431b7b40391599cb1c5']
New Aggregated Model - Loss: 9.432328224182129, Accuracy: 0.4487000107765198


2026/09/03 12:37:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Previous Aggregated Model - Loss: 10.084258079528809, Accuracy: 0.4077000021934509


2026/09/03 12:37:06 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run deltarep_cycle3 at: http://localhost:5100/#/experiments/1/runs/7750e59a74ce4508bbabdcfd019529a1
🧪 View experiment at: http://localhost:5100/#/experiments/1
7750e59a74ce4508bbabdcfd019529a1


In [30]:
!docker stop aggregator_test

aggregator_test
